# Monitoring Module Demo

This notebook demonstrates the Monitoring Module components working together in a simulated production stream:

1. **RealTimeFairnessTracker**: Processes batches over time and builds time-series metrics
2. **FairnessDriftAndAlertEngine**: Detects drift and generates prioritized alerts
3. **FairnessReportingDashboard**: Visualizes trends and intersectional metrics
4. **FairnessABTestAnalyzer**: Demonstrates A/B testing capabilities (optional)

We'll simulate a production stream where fairness characteristics change over time, triggering drift detection and alerts.


In [ ]:
import numpy as np
import pandas as pd
import time
from datetime import datetime, timedelta
import os

from fairness_pipeline_dev_toolkit.monitoring import (
    RealTimeFairnessTracker,
    ColumnMap,
    TrackerConfig,
    FairnessDriftAndAlertEngine,
    FairnessReportingDashboard,
    MonitoringSettings,
    FairnessABTestAnalyzer,
)

print("Monitoring module imports successful!")


## Setup: Initialize Monitoring Components

We'll set up:
- A tracker with a sliding window
- A drift detection engine with configurable thresholds
- A dashboard for visualization


In [ ]:
# Create artifacts directory
artifacts_dir = "artifacts/monitoring_demo"
os.makedirs(artifacts_dir, exist_ok=True)

# Configure tracker: sliding window of 5000 samples, min group size 20
tracker_cfg = TrackerConfig(
    window_size=5000,
    min_group_size=20,
    metrics=("demographic_parity", "equalized_odds")
)

# Initialize tracker
tracker = RealTimeFairnessTracker(tracker_cfg, artifacts_dir=artifacts_dir)

# Configure drift detection
monitoring_settings = MonitoringSettings(
    artifacts_dir=artifacts_dir,
    drift=MonitoringSettings().drift  # Use defaults
)
drift_engine = FairnessDriftAndAlertEngine(monitoring_settings)

# Initialize dashboard
dashboard = FairnessReportingDashboard(monitoring_settings)

print(f"✅ Monitoring components initialized")
print(f"   Artifacts directory: {artifacts_dir}")
print(f"   Tracker window size: {tracker_cfg.window_size}")
print(f"   Min group size: {tracker_cfg.min_group_size}")


## Simulate Production Stream

We'll generate batches over time with varying fairness characteristics:
- **Phase 1 (baseline)**: Fair model with balanced predictions across groups
- **Phase 2 (drift)**: Model starts showing bias against group "C"
- **Phase 3 (severe drift)**: Significant fairness degradation
- **Phase 4 (recovery)**: Model improves but still has some bias


In [ ]:
def generate_batch(n_samples, group_probs, pred_probs_by_group, seed_offset=0):
    """
    Generate a batch of predictions with specified group distributions and prediction rates.
    
    Parameters:
    - n_samples: number of samples in batch
    - group_probs: probability distribution for groups (e.g., {"A": 0.5, "B": 0.3, "C": 0.2})
    - pred_probs_by_group: probability of positive prediction per group (e.g., {"A": 0.5, "B": 0.5, "C": 0.5})
    - seed_offset: for reproducibility
    """
    rng = np.random.default_rng(42 + seed_offset)
    
    # Sample groups
    groups = list(group_probs.keys())
    probs = list(group_probs.values())
    group_col = rng.choice(groups, size=n_samples, p=probs)
    
    # Generate ground truth (somewhat correlated with group for realism)
    y_true = rng.integers(0, 2, size=n_samples)
    
    # Generate predictions based on group-specific probabilities
    y_pred_probs = np.array([pred_probs_by_group[g] for g in group_col])
    y_pred = (rng.random(n_samples) < y_pred_probs).astype(int)
    
    # Add some correlation between predictions and ground truth
    y_pred = np.where(rng.random(n_samples) < 0.7, y_true, y_pred)
    
    # Create DataFrame
    df = pd.DataFrame({
        "y_pred": y_pred,
        "y_true": y_true,
        "gender": group_col,
        "score": rng.random(n_samples)  # probability scores
    })
    
    return df

print("✅ Batch generation function ready")


In [ ]:
# Define phases of the production stream
phases = [
    {
        "name": "Phase 1: Baseline (Fair)",
        "n_batches": 10,
        "batch_size": 200,
        "group_probs": {"A": 0.5, "B": 0.3, "C": 0.2},
        "pred_probs": {"A": 0.5, "B": 0.5, "C": 0.5},  # Fair predictions
    },
    {
        "name": "Phase 2: Initial Drift",
        "n_batches": 8,
        "batch_size": 200,
        "group_probs": {"A": 0.5, "B": 0.3, "C": 0.2},
        "pred_probs": {"A": 0.55, "B": 0.5, "C": 0.4},  # Group C gets fewer positives
    },
    {
        "name": "Phase 3: Severe Drift",
        "n_batches": 6,
        "batch_size": 200,
        "group_probs": {"A": 0.5, "B": 0.3, "C": 0.2},
        "pred_probs": {"A": 0.6, "B": 0.5, "C": 0.3},  # Strong bias against C
    },
    {
        "name": "Phase 4: Partial Recovery",
        "n_batches": 8,
        "batch_size": 200,
        "group_probs": {"A": 0.5, "B": 0.3, "C": 0.2},
        "pred_probs": {"A": 0.52, "B": 0.5, "C": 0.45},  # Improved but still biased
    },
]

print("✅ Production stream phases defined")
for phase in phases:
    print(f"   {phase['name']}: {phase['n_batches']} batches × {phase['batch_size']} samples")


In [ ]:
# Process batches through the tracker
print("Processing production stream...")
print("-" * 60)

batch_count = 0
column_map = ColumnMap(
    y_pred="y_pred",
    y_true="y_true",
    protected=["gender"],
    intersections=[]  # Can add intersections like [["gender", "age_group"]]
)

for phase_idx, phase in enumerate(phases):
    print(f"\n{phase['name']}")
    print(f"  Processing {phase['n_batches']} batches...")
    
    for batch_idx in range(phase['n_batches']):
        batch = generate_batch(
            n_samples=phase['batch_size'],
            group_probs=phase['group_probs'],
            pred_probs_by_group=phase['pred_probs'],
            seed_offset=batch_count
        )
        
        # Process batch through tracker
        window = tracker.process_batch(batch, column_map)
        batch_count += 1
        
        # Small delay to simulate real-time processing
        time.sleep(0.1)
    
    print(f"  ✅ Completed {phase['n_batches']} batches")

print(f"\n✅ Total batches processed: {batch_count}")
print(f"✅ Metrics time series shape: {tracker.metrics_ts.shape}")
print(f"✅ Metrics stored with DatetimeIndex: {isinstance(tracker.metrics_ts.index, pd.DatetimeIndex)}")


## Inspect Metrics Time Series

Let's examine the collected metrics:


In [ ]:
# Display metrics time series
print("Metrics Time Series (first 20 rows):")
print(tracker.metrics_ts.head(20))
print(f"\nTotal metric points: {len(tracker.metrics_ts)}")
print(f"\nUnique metrics: {tracker.metrics_ts['metric'].unique()}")
print(f"\nUnique groups: {tracker.metrics_ts['group_key'].unique()}")
print(f"\nTime range: {tracker.metrics_ts.index.min()} to {tracker.metrics_ts.index.max()}")


## Detect Drift and Generate Alerts

The drift engine analyzes the time series and identifies significant changes:


In [ ]:
# Analyze drift with smaller windows for demo (fewer points needed)
alerts = drift_engine.analyze(
    tracker.metrics_ts,
    window_points=5,  # Recent window
    ref_points=15    # Reference window
)

print(f"✅ Drift analysis complete")
print(f"   Total alerts generated: {len(alerts)}")

if alerts:
    print("\nAlert Details:")
    print("-" * 80)
    for alert in alerts:
        print(f"  [{alert.severity}] {alert.metric} - {alert.group_key}")
        print(f"    Timestamp: {alert.timestamp}")
        print(f"    Drift Score: {alert.drift_score:.4f}")
        print(f"    Reason: {alert.reason}")
        print()
else:
    print("   No alerts triggered (drift thresholds not exceeded)")


## Visualize Trends

The dashboard provides interactive visualizations of fairness metrics over time:


In [ ]:
# Plot demographic parity trends
fig_dp = dashboard.plot_trend(tracker.metrics_ts, "DP[gender]")
fig_dp.show()


In [ ]:
# Plot equalized odds trends
fig_eo = dashboard.plot_trend(tracker.metrics_ts, "EO[gender]")
fig_eo.show()


## Intersectional Heatmap

The heatmap visualization shows fairness metrics across intersectional subgroups:


In [ ]:
# Plot intersectional heatmap (latest snapshot)
fig_heatmap = dashboard.plot_intersectional(tracker.metrics_ts, "DP[", latest_only=True)
fig_heatmap.show()


## Generate Monitoring Report

Create a markdown report summarizing the monitoring session:


In [ ]:
# Convert alerts to dict format for report
alerts_dict = [alert.__dict__ for alert in alerts]

# Generate markdown report
report_path = dashboard.write_markdown_report(
    tracker.metrics_ts,
    alerts_dict,
    name="monitoring_report.md",
    summary_title="Fairness Monitoring Report - Production Stream Demo"
)

print(f"✅ Report generated: {report_path}")

# Display report
with open(report_path, 'r') as f:
    print("\n" + "=" * 80)
    print("MONITORING REPORT")
    print("=" * 80)
    print(f.read())


## Optional: A/B Testing Analysis

Demonstrate A/B testing capabilities with the FairnessABTestAnalyzer:


In [ ]:
# Simulate control and treatment groups for A/B test
# Control: baseline fair model
# Treatment: model with some bias

rng = np.random.default_rng(42)
n_ab = 500

# Control group (fair)
control_df = pd.DataFrame({
    "y_pred": rng.integers(0, 2, n_ab),
    "y_true": rng.integers(0, 2, n_ab),
    "gender": rng.choice(["A", "B", "C"], n_ab, p=[0.5, 0.3, 0.2]),
    "fairness_metric": rng.random(n_ab) * 0.1,  # Low disparity
})

# Treatment group (biased)
treatment_df = pd.DataFrame({
    "y_pred": np.where(rng.choice(["A", "B", "C"], n_ab, p=[0.5, 0.3, 0.2]) == "C",
                       (rng.random(n_ab) < 0.3).astype(int),  # Lower rate for C
                       rng.integers(0, 2, n_ab)),
    "y_true": rng.integers(0, 2, n_ab),
    "gender": rng.choice(["A", "B", "C"], n_ab, p=[0.5, 0.3, 0.2]),
    "fairness_metric": np.where(rng.choice(["A", "B", "C"], n_ab, p=[0.5, 0.3, 0.2]) == "C",
                                rng.random(n_ab) * 0.3,  # Higher disparity for C
                                rng.random(n_ab) * 0.1),
})

# Initialize A/B test analyzer
ab_analyzer = FairnessABTestAnalyzer(
    control=control_df,
    treatment=treatment_df,
    protected_attributes=["gender"],
    outcome_column="y_pred",
    fairness_metric_column="fairness_metric",
    business_metrics=[]
)

# Power analysis by intersection
from fairness_pipeline_dev_toolkit.monitoring.abtest import ABPowerSpec
power_spec = ABPowerSpec(effect_size=0.10, alpha=0.05, min_group_size=30)
power_results = ab_analyzer.power_by_intersection(power_spec)

print("A/B Test Power Analysis by Intersection:")
print("-" * 60)
for intersection, power in power_results.items():
    print(f"  {intersection}: {power:.3f}" if not np.isnan(power) else f"  {intersection}: N/A (insufficient sample)")

# Heterogeneous treatment effects
effects = ab_analyzer.heterogeneous_effects(n_bootstrap=500, alpha=0.05)
print("\nHeterogeneous Treatment Effects:")
print("-" * 60)
print(effects[effects["metric"] == "fairness_metric"])
